# S3·LUN — El loop de tool use, desde cero

**MMIA 6013 · Semana 3 · Lunes**

Construyes la maquinaria completa de un agente: definición de tools, el loop
decidir→ejecutar→observar, y la traza de cada paso.

Sin API key, un `LLMSimulado` emite las mismas estructuras `tool_use` que un
modelo real: **el loop que escribes es idéntico en ambos casos** — esa es la
gracia del ejercicio. Con clave, el mismo loop corre contra OpenAI con
*function calling* de verdad; y si hay VPN, contra la H200 de la universidad
sin gastar un centavo.

In [ ]:
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path


def _raiz_repo(inicio: Path) -> Path:
    """Sube directorios hasta la raíz del repositorio del curso.

    La ruta relativa fija falla: desde `curso/actividades/semana-N/` dos niveles
    arriba es `curso/`, no la raíz. Buscar hacia arriba hace además que el
    cuaderno corra desde donde se lo abra.
    """
    for base in [inicio, *inicio.parents]:
        if (base / "curso/actividades").is_dir():
            return base
    raise FileNotFoundError(
        "No encuentro `curso/actividades/`: abre el cuaderno dentro del "
        "repositorio del curso.")


RAIZ = _raiz_repo(Path.cwd())

# La clave se lee del `.env` de la raíz. NUNCA se escribe aquí: la política de
# credenciales lo prohíbe, y una salida guardada en el .ipynb es justo el sitio
# donde no puede aparecer.
_env = RAIZ / ".env"
if _env.is_file():
    for _linea in _env.read_text(encoding="utf-8").splitlines():
        _linea = _linea.strip()
        if _linea and not _linea.startswith("#") and "=" in _linea:
            _k, _, _v = _linea.removeprefix("export ").partition("=")
            _k, _v = _k.strip(), _v.strip().strip('"').strip("'")
            if _k and _v:
                os.environ.setdefault(_k, _v)   # lo exportado en la terminal manda

MODELO_API = "gpt-5.6-terra"       # un solo sitio donde cambiarlo
MAX_SALIDA = 1200                  # el presupuesto incluye el razonamiento
H200 = "http://172.28.230.10:12555/v1"
MODELO_H200 = "zai-org/GLM-5.3-Flash"

print(f"raíz del curso : {RAIZ}")
print(f"OPENAI_API_KEY : {'presente' if os.environ.get('OPENAI_API_KEY') else 'ausente'}")

# Dataset en memoria: ventas de una empresa ficticia
VENTAS = [
    {"fecha": "2026-03-02", "producto": "laptop", "region": "sierra", "monto": 1200},
    {"fecha": "2026-03-05", "producto": "monitor", "region": "costa", "monto": 300},
    {"fecha": "2026-03-11", "producto": "laptop", "region": "costa", "monto": 1150},
    {"fecha": "2026-03-15", "producto": "teclado", "region": "sierra", "monto": 45},
    {"fecha": "2026-04-01", "producto": "laptop", "region": "oriente", "monto": 1300},
    {"fecha": "2026-04-03", "producto": "monitor", "region": "sierra", "monto": 320},
    {"fecha": "2026-04-18", "producto": "teclado", "region": "costa", "monto": 50},
]

## Parte 1 — Definir las tools

Cada tool = función Python + esquema (nombre, descripción, parámetros).
**La descripción es el prompt:** el modelo elige la tool leyéndola.

In [ ]:
def consultar_ventas(mes: str, region: str | None = None) -> str:
    filas = [v for v in VENTAS if v["fecha"].startswith(mes)
             and (region is None or v["region"] == region)]
    return json.dumps(filas, ensure_ascii=False)

def estadisticas_ventas(mes: str) -> str:
    montos = [v["monto"] for v in VENTAS if v["fecha"].startswith(mes)]
    if not montos:
        return json.dumps({"error": f"sin datos para {mes}"})
    return json.dumps({"n": len(montos), "total": sum(montos),
                       "promedio": round(sum(montos) / len(montos), 2)})

def listar_productos() -> str:
    return json.dumps(sorted({v["producto"] for v in VENTAS}))

ESQUEMAS = [
    {
        "name": "consultar_ventas",
        "description": ("Devuelve las filas de ventas de un mes dado, opcionalmente filtradas "
                        "por región (sierra, costa u oriente). Úsala cuando el usuario pida "
                        "el detalle de ventas. No calcula agregados."),
        "input_schema": {"type": "object",
                         "properties": {
                             "mes": {"type": "string", "description": "Mes en formato YYYY-MM"},
                             "region": {"type": "string", "enum": ["sierra", "costa", "oriente"]}},
                         "required": ["mes"]},
    },
    {
        "name": "estadisticas_ventas",
        "description": ("Calcula número de ventas, total y promedio de un mes. Úsala cuando "
                        "pidan cifras agregadas ('cuánto vendimos', 'promedio')."),
        "input_schema": {"type": "object",
                         "properties": {"mes": {"type": "string", "description": "YYYY-MM"}},
                         "required": ["mes"]},
    },
    {
        "name": "listar_productos",
        "description": "Lista los productos distintos que aparecen en las ventas.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

TOOLS = {"consultar_ventas": consultar_ventas,
         "estadisticas_ventas": estadisticas_ventas,
         "listar_productos": listar_productos}

### El mismo contrato, dos formatos de cable

`ESQUEMAS` está escrito en el formato de **Anthropic**: `name`, `description` e
`input_schema`. OpenAI pide exactamente la misma información con otra envoltura:
`{"type": "function", "function": {"name", "description", "parameters"}}`.

No es una anécdota de sintaxis. Es **la razón por la que existe MCP**, que se ve el
miércoles: la herramienta es la misma, la descripción es la misma, el esquema JSON es el
mismo, y cada proveedor la quiere envuelta a su manera. Cada integración nueva multiplica
ese trabajo por el número de proveedores. MCP propone declararla una vez.

Quedarse con esta idea hoy hace que el miércoles no sea una API más que aprender.

In [ ]:
def a_formato_openai(esquemas):
    """Traduce los esquemas al formato de OpenAI. La información no cambia; la caja sí."""
    return [{"type": "function",
             "function": {"name": e["name"],
                          "description": e["description"],
                          "parameters": e.get("input_schema", {"type": "object",
                                                               "properties": {}})}}
            for e in esquemas]


ESQUEMAS_OPENAI = a_formato_openai(ESQUEMAS)

print("Anthropic:", json.dumps(ESQUEMAS[1], ensure_ascii=False)[:150], "…")
print("OpenAI   :", json.dumps(ESQUEMAS_OPENAI[1], ensure_ascii=False)[:150], "…")
print(f"\nmisma información, {len(ESQUEMAS)} herramientas, dos envolturas")

## Parte 2 — El "LLM": simulado o real

Interfaz común: recibe la conversación (`messages`) y devuelve o bien
`{"tipo": "tool_use", "name": ..., "input": ...}` o bien
`{"tipo": "texto", "texto": ...}`.

In [ ]:
class LLMSimulado:
    """Sigue un guion por palabras clave. Emite las MISMAS estructuras que un
    modelo real con function calling — el loop no nota la diferencia."""

    def responder(self, messages):
        pregunta = messages[0]["content"].lower()
        ya_llamadas = [m["name"] for m in messages if m.get("role") == "tool_result_meta"]

        if "cuánto" in pregunta or "total" in pregunta or "promedio" in pregunta:
            if "estadisticas_ventas" not in ya_llamadas:
                mes = "2026-03" if "marzo" in pregunta else "2026-04"
                return {"tipo": "tool_use", "name": "estadisticas_ventas", "input": {"mes": mes}}
            datos = json.loads(messages[-1]["content"])
            return {"tipo": "texto",
                    "texto": f"En ese mes hubo {datos['n']} ventas por un total de "
                             f"${datos['total']} (promedio ${datos['promedio']})."}

        if "productos" in pregunta:
            if "listar_productos" not in ya_llamadas:
                return {"tipo": "tool_use", "name": "listar_productos", "input": {}}
            return {"tipo": "texto", "texto": f"Los productos son: {messages[-1]['content']}"}

        if "detalle" in pregunta or "ventas de" in pregunta:
            if "consultar_ventas" not in ya_llamadas:
                entrada = {"mes": "2026-03" if "marzo" in pregunta else "2026-04"}
                if "sierra" in pregunta:
                    entrada["region"] = "sierra"
                return {"tipo": "tool_use", "name": "consultar_ventas", "input": entrada}
            return {"tipo": "texto", "texto": f"Detalle encontrado: {messages[-1]['content']}"}

        return {"tipo": "texto", "texto": "¿Puedes reformular? Puedo consultar ventas, estadísticas o productos."}


class LLMReal:
    """El mismo `responder(messages)`, contra un servicio que habla el protocolo de OpenAI.

    Traduce el historial interno —`assistant_tool_use` y `tool_result`— al del cable, que
    es distinto y conviene mirarlo de cerca:

      · la petición del modelo va en `message.tool_calls`, con los argumentos **como
        cadena JSON**, no como objeto: hay que parsearlos, y un modelo puede devolver JSON
        inválido ahí. Es el primer sitio donde un agente se rompe en producción.
      · el resultado vuelve en un mensaje de rol `tool` con el `tool_call_id` que emparejó.
        Ese emparejamiento es la invariante (2) del loop, escrita en el protocolo.
    """

    def __init__(self, base_url, modelo, api_key=None, etiqueta="API"):
        self.base_url, self.modelo, self.api_key, self.etiqueta = base_url, modelo, api_key, etiqueta

    def _pedir(self, mensajes):
        cuerpo = {"model": self.modelo, "messages": mensajes,
                  "tools": ESQUEMAS_OPENAI, "tool_choice": "auto"}
        # `max_completion_tokens` en OpenAI; el vLLM de la H200 usa `max_tokens`. Mandar el
        # que no toca devuelve 400 en OpenAI, así que se elige por destino.
        cuerpo["max_completion_tokens" if self.api_key else "max_tokens"] = MAX_SALIDA
        cabeceras = {"Content-Type": "application/json"}
        if self.api_key:
            cabeceras["Authorization"] = f"Bearer {self.api_key}"
        pet = urllib.request.Request(f"{self.base_url}/chat/completions",
                                     data=json.dumps(cuerpo).encode(), headers=cabeceras)
        with urllib.request.urlopen(pet, timeout=120) as r:
            return json.loads(r.read())

    def responder(self, messages):
        cable = [{"role": "user", "content": messages[0]["content"]}]
        for m in messages[1:]:
            if m["role"] == "assistant_tool_use":
                cable.append({"role": "assistant", "content": None, "tool_calls": [
                    {"id": m["id"], "type": "function",
                     "function": {"name": m["name"], "arguments": json.dumps(m["input"])}}]})
            elif m["role"] == "tool_result":
                cable.append({"role": "tool", "tool_call_id": m["id"], "content": m["content"]})

        respuesta = self._pedir(cable)
        uso = respuesta.get("usage", {})
        msg = respuesta["choices"][0]["message"]
        print(f"   [{self.etiqueta}] {uso.get('prompt_tokens')} tok entrada / "
              f"{uso.get('completion_tokens')} salida")
        llamadas = msg.get("tool_calls") or []
        if llamadas:
            fn = llamadas[0]["function"]
            try:
                entrada = json.loads(fn["arguments"] or "{}")
            except json.JSONDecodeError:
                # No se inventa un valor: se devuelve el error como texto y el loop termina.
                return {"tipo": "texto",
                        "texto": f"El modelo pidió {fn['name']} con argumentos que no son "
                                 f"JSON válido: {fn['arguments']!r}"}
            return {"tipo": "tool_use", "name": fn["name"], "input": entrada,
                    "id": llamadas[0]["id"]}
        return {"tipo": "texto", "texto": (msg.get("content") or "").strip()}


def hacer_llm():
    """Tres peldaños, del más capaz al que siempre funciona, y dice cuál eligió."""
    clave = os.environ.get("OPENAI_API_KEY")
    if clave:
        try:
            llm = LLMReal("https://api.openai.com/v1", MODELO_API, clave, "API")
            llm._pedir([{"role": "user", "content": "ping"}])       # una prueba, no una fe
            print(f"LLM: OpenAI · {MODELO_API}")
            return llm
        except urllib.error.HTTPError as e:
            print(f"LLM: OpenAI rechazó la petición ({e.code} · {e.read().decode()[:120]})")
        except urllib.error.URLError as e:
            print(f"LLM: sin red hacia OpenAI ({e.reason})")
    try:
        llm = LLMReal(H200, MODELO_H200, None, "H200")
        llm._pedir([{"role": "user", "content": "ping"}])
        print(f"LLM: H200 de la USFQ · {MODELO_H200} (gratis)")
        return llm
    except Exception as e:
        print(f"LLM: la H200 no responde ({type(e).__name__}) — ¿VPN?")
    print("LLM: simulado (determinista, sin costo). El loop es el mismo.")
    return LLMSimulado()


llm = hacer_llm()

## Parte 3 — EL LOOP (tu turno)

### ✏️ Ejercicio 3.1
Implementa `ejecutar_agente(pregunta, max_pasos=5)`:
1. Inicia `messages = [{"role": "user", "content": pregunta}]` y `traza = []`
2. En cada paso: pide respuesta al `llm`.
   - Si es `texto` → registra en la traza y devuélvelo (fin).
   - Si es `tool_use` → ejecuta `TOOLS[name](**input)`, registra el evento
     en la traza, y añade al historial el tool_use y su tool_result.
3. Si llegas a `max_pasos` sin respuesta final, corta y devuélvelo indicado.
   (¡Todo agente necesita un límite de pasos: es tu seguro anti-loop!)

In [ ]:
# Ejercicio 3.1 — el corazón de todo agente

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

## Parte 4 — La traza: tu caja negra de vuelo

Cada corrida deja una lista de eventos. Inspecciónala: es lo que en la
Semana 4 enviaremos a Langfuse.

In [ ]:
for e in traza:
    print(e)

## Parte 4 — Leer la traza

### ✏️ Ejercicio 4.1
Corre el agente con estas tres preguntas y contesta en un comentario:

1. «¿Cuánto vendimos en marzo?» — una sola herramienta
2. «¿Qué productos tenemos?» — otra herramienta, sin argumentos
3. «¿Cuántos pasos tarda si la pregunta no se puede contestar con las tres?»

Para cada una, mira la **traza**: cuántos pasos dio, qué herramienta pidió, con qué
argumentos, y en qué paso decidió que ya tenía lo necesario. Lo que hay que notar no es la
respuesta final: es que el LLM **nunca ejecutó nada** —solo pidió— y que cada `tool_use`
recibió exactamente un `tool_result`. Esas dos invariantes son todo el contrato del loop.

In [ ]:
# Ejercicio 4.1 — corre estas tres preguntas y responde en el comentario final:
#   1. "¿Qué productos vendemos?"
#   2. "Dame el detalle de ventas de marzo en sierra"
#   3. "¿Cuál es el sentido de la vida?"   ← ¿qué hace el agente sin tool adecuada?

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

In [ ]:
# 🤔 Reflexión final (comentario):
# a) ¿Qué pasó con la pregunta 3 (sin tool adecuada)? ¿Es el comportamiento correcto?
# b) En el loop, ¿qué protecciones añadirías antes de darle a este agente una
#    tool que ESCRIBE en la base de datos? (mínimo dos)
#
# a)
# b)

## Parte 5 — La taxonomía de la sala de juntas

Fuera de esta clase vas a encontrar la IA explicada en capas. Una versión que
circula mucho en ámbito ejecutivo, y que te van a preguntar en el trabajo:

| Capa | Qué hace | Herramientas que se citan |
|---|---|---|
| 1 · Machine Learning | predice sobre datos históricos | SageMaker, Vertex AI |
| 2 · IA Generativa | genera contenido a partir de un prompt | ChatGPT, Claude, Gemini |
| 3 · Agentes de IA | ejecuta tareas de varios pasos con herramientas | LangChain, CrewAI |
| 4 · Agentic AI | orquesta procesos de negocio completos | Claude Code, Codex |

Es un buen mapa para una conversación de negocio. La pregunta es qué tan bien
aguanta cuando la contrastas con el código que acabas de escribir.

In [ ]:
# ✏️ Ejercicio 5.1 (comentario)
#
# a) El loop que escribiste en la Parte 3: ¿es capa 3 o capa 4?
# b) Para pasar de una a otra, ¿qué línea de TU código cambiaría?
# c) Entonces, ¿qué distingue realmente la capa 3 de la capa 4?
# d) Según este mapa, un chatbot que responde con un prompt largo y sin tools
#    ¿en qué capa está? ¿Y según la definición operativa de esta clase?
#
# a)
# b)
# c)
# d)

## Cierre

- El agente = LLM que decide + tu código que ejecuta + loop con límite de pasos.
- La descripción de la tool es el prompt que gobierna la elección.
- Toda corrida deja traza — sin traza no hay debugging ni observabilidad.
- Un agente se define por la decisión, no por la escala: sin elección de tool
  en bucle no hay agente, por muchas capas que le ponga el diagrama.

**Mañana:** patrones de razonamiento del agente — ReAct, Plan-and-Execute y
Reflexion — y cuándo usar cada uno.